### Tockeization

Tokenization = splitting text into words




**This decision affects:**

* model performance
* training cost
* multilingual capability
* reasoning ability
* context length
* efficiency

**Why Do We Need Tokenization?**

Computers cannot directly process: "The cat sat on the mat"

Transformers require integers.

We need a mapping: Text
↓
Tokens
↓
Token IDs
↓
Embeddings



**Example:**

"The cat sat" becomes  --> ["The", "cat", "sat"] --> becomes --> [101, 205, 876]



**Why Not Character-Level Tokenization?**

Example: cat -- > becomes --> [c, a, t]

Advantages:
tiny vocabulary ,  handles every word

Problems:
* sequences become very long
* training becomes expensive
* harder to learn semantics


**Example:**

internationalization --> 21 characters

One tokenization decision becomes: 21 separate tokens

Attention cost grows quadratically.


**Subword Tokenization (Modern Solution)**

Modern LLMs use: Subword Tokenization

**Idea:** Split words into reusable pieces. Example:playing   --> becomes: play,ing

**Example:** unhappiness   --> becomes: un,happi,ness

**Benefits:**
* manageable vocabulary
* shorter sequences than characters
* This is why almost every modern LLM uses subword tokenization.


# How Positional Encoding is Assigned Initially
Because the Transformer architecture processes all words in parallel (unlike RNNs which read word-by-word), it has no inherent sense of order. We must inject position information. There 

are three main ways this is initialized:

## 1. Fixed Sinusoidal Encoding (The Original Transformer)
In the original 2017 "Attention Is All You Need" paper, positional encodings are not learned. They are hardcoded using sine and cosine waves of different frequencie.

The Math: For a given position pos and dimension  i, the value is calculated as:

$$
PE(pos,2i)
=
\sin
\left(
\frac{pos}
{10000^{\frac{2i}{d_{model}}}}
\right)
$$

$$
PE(pos,2i+1)
=
\cos
\left(
\frac{pos}
{10000^{\frac{2i}{d_{model}}}}
\right)
$$

**How it's assigned:** 

At initialization, a matrix of size **[*max_sequence_length*, *embedding_dimension*]** is created. Every single cell is filled using the sine/cosine formulas above.

**Why?**
It allows the model to learn relative positions between tokens. For example, the model can recognize that a token at position 5 is two positions away from a token at position 3. This is possible because the sinusoidal encodings satisfy the property that:

$$
PE_{pos+k}
$$

can be expressed as a linear function of

$$
PE_{pos}
$$

As a result, the Transformer can infer relative distances between tokens directly from their positional encodings, making it easier to model word order and long-range dependencies.




## 2. Learned Positional Embeddings (GPT-2, BERT)

Instead of using math formulas, the model just creates a standard matrix of random numbers.

**How it's assigned:** 
A matrix of size **[*max_sequence_length, embedding_dimension*]** is initialized using a random normal distribution (e.g., mean=0, std=0.02).

**What happens next:**

During training, backpropagation updates these numbers. 
The model learns the best way to represent position 1, position 2, etc.



## 3. Rotary Position Embedding / RoPE (The Modern Standard: Llama, Mistral, Qwen)

Modern LLMs rarely add positional encodings to the word embeddings anymore. Instead, they use RoPE.

**How it's assigned:** RoPE does not create a separate positional vector. Instead, it creates a fixed rotation matrix based on trigonometric functions (similar to the original sinusoidal math).

$$
\theta_i(pos)
=
\frac{pos}
{10000^{\frac{2i}{d_{model}}}}
$$

- **What happens next:** When the input is multiplied by the $W_Q$ and $W_K$ matrices to create Queries and Keys, those resulting vectors are rotated in multi-dimensional space based on their position.

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

The rotated Query and Key vectors become:

$$
Q' = R(pos)Q
$$

$$
K' = R(pos)K
$$

The Value matrix is not rotated:

$$
V' = V
$$

### Types of Teockenization 

**The 3 Main Subword Algorithms:**

**BPE (Byte-Pair Encoding)**: Starts small and merges the most frequently occurring pairs of  characters. (Used by GPT, Llama, Mistral).

**WordPiece:** Starts small and merges pairs that maximize the probability of the training data Uses to mark subword continuations. (Used by BERT)

**Unigram:** Starts with a massive vocabulary and iteratively removes the least useful tokens until it reaches the target size. (Used by T5, ALBERT, Gemini).


### The Software Libraries  

**tiktoken:** OpenAI’s highly optimized library (used for GPT models).

**SentencePiece:** Google’s go-to library that handles both BPE and Unigram (used for Gemini, T5, Llama).

**Proprietary/Custom:** Models like Claude and DeepSeek use closed-source or custom BPE implementations, often approximated by community wrappers.

### Byte Pair Encoding (BPE)  


One of the most important tokenization algorithms.

Used by: GPT-2, GPT-3 , many LLMs


#### Why Was BPE Created?

Suppose your corpus contains:

* cat
* cats
* dog
* dogs
* play
* playing
* played
* player


If we use word-level tokenization, vocabulary becomes:

* cat
* cats
* dog
* dogs
* play
* playing
* played
* player
* ...

Eventually millions of words appear.

Problems:
* huge vocabulary
* memory expensive
* unseen words fail

##Suppose corpus is: Core Idea of BPE

BPE starts with: Characters

Then repeatedly merges the most frequent adjacent pair.

Think: c a t

becomes

c a t --> ca t  ---> then cat

The vocabulary grows gradually.



### Step 1: Training Corpus

Suppose corpus is:

* cat
* cat
* cats
* cats
* dog
* dog
* dogs

**Initially every word is split into characters.**

c a t

c a t

c a t s

c a t s

d o g

d o g

d o g s



Initial vocabulary:

c

a

t

s

d

o

g


**Vocabulary size:**

7



##  2: Count Adjacent Pairs

Look at every neighboring pair.

**Corpus:**

c a t

c a t

c a t s

c a t s




**Pairs:**

(c,a)

(a,t)

(t,s)


**Count frequencies.**

Below is the frequency distribution of the adjacent character pairs in our corpus before applying the BPE merges:

| Pair  | Frequency |
| :---  | :-------- |
| `c a` | 4 |
| `a t` | 4 |
| `t s` | 2 |
| `d o` | 3 |
| `o g` | 3 |
| `g s` | 1 |

Most frequent: c a  and frequency: 4

## Step 3: Merge Most Frequent Pair

Merge:

c a

into:

ca


**Corpus becomes:**

ca t

ca t

ca t s

ca t s

d o g

d o g

d o g s


**Vocabulary becomes:**

c

a

t

s

d

o

g

ca

**Vocabulary size: 8** Note ca is added as extra 

## Step 4: Repeat

Recompute frequencies.

Now pairs:

(ca,t)

(t,s)

(d,o)

(o,g)



#### Step 5: Identify the Next Merge
We evaluate the updated frequencies to find the next pair to merge.

| Pair | Frequency |
| :--- | :-------- |
| `ca t` | 4 |
| `d o`  | 3 |
| `o g`  | 3 |
| `t s`  | 2 |

**Most Frequent Pair:** `ca t` (Count: 4)

> **Action:** The algorithm selects `ca t` and merges it into the new token **`cat`**. This demonstrates how BPE progressively builds complete, meaningful words from smaller character chunks.



## Step 5: Continuing the BPE Merges
After successfully forming the token `cat`, the algorithm evaluates the remaining corpus to find the next most frequent pair.

**Current Corpus State:**
```text
cat
cat
cat s
cat s
d o g
d o g
d o g s



Current Vocabulary: ['c', 'a', 't', 's', 'd', 'o', 'g', 'ca', 'cat']

Action: The most frequent remaining pair is d o (appearing 3 times). The algorithm merges d and o into do, and subsequently merges do and g into dog.



## Step 6: More Merges

**Frequent pairs:**

cat s
dog s



**Merge:**

cats
dogs

**Eventually vocabulary becomes:**

* cat
* cat
* dog
* dogs




## Embedding Matrix


$E \in  R^{Vd}$



where: V = vocabulary size , d = embedding dimension



"cat sat on mat" --> ["cat","sat","on","mat"] -- > [12,45,78,33]


#### Embedding Lookup

We have: [12,45,78,33]

These numbers are only addresses.

For token: 12

Model does: embedding = E[12]

Result:

$X_{cat} = [0.2, 0.8, 0.1, 0.5]$


$X_{sat}​=[0.9,0.3,0.4,0.1]$

$X_{on}​=[0.5,0.2,0.7,0.3]$


$X_{mat}​=[0.6,0.9,0.2,0.4]$

## Input Matrix X

Now stack them together.

$$
X=
\begin{bmatrix}
0.2 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.3 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.7 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.4
\end{bmatrix}
$$



**Shape:**

4×4 Interpretation: 4 tokens, 4-dimensional embedding

### Positional Encoding

Transformer processes everything simultaneously.

Current matrix: X

contains:

cat

sat

on

mat


But it doesn't know:   cat sat on mat  versus   mat on sat cat

So position vectors are added.

**Example:**

Position 1:

$P_1 =[0.1,0,0,0]$ 

Position 2:

$P_2 =[0,0.1,0,0]$

Position 3

$P_3 =[0,0,0.1,0]$

Position 4:

$P_4 =[0,0,0,0.1]$ 

## Positional Encoding

After obtaining the embedding matrix:

$$
X=
\begin{bmatrix}
0.2 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.3 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.7 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.4
\end{bmatrix}
$$

we add positional vectors to encode token order.

Assume:

$$
P=
\begin{bmatrix}
0.1 & 0.0 & 0.0 & 0.0 \\
0.0 & 0.1 & 0.0 & 0.0 \\
0.0 & 0.0 & 0.1 & 0.0 \\
0.0 & 0.0 & 0.0 & 0.1
\end{bmatrix}
$$

The position-aware representation is computed as:

$$
X_{pos}=X+P
$$

For the token cat:

$$
[0.2,0.8,0.1,0.5]
+
[0.1,0.0,0.0,0.0]
=
[0.3,0.8,0.1,0.5]
$$

For the token sat:

$$
[0.9,0.3,0.4,0.1]
+
[0.0,0.1,0.0,0.0]
=
[0.9,0.4,0.4,0.1]
$$

For the token on:

$$
[0.5,0.2,0.7,0.3]
+
[0.0,0.0,0.1,0.0]
=
[0.5,0.2,0.8,0.3]
$$

For the token mat:

$$
[0.6,0.9,0.2,0.4]
+
[0.0,0.0,0.0,0.1]
=
[0.6,0.9,0.2,0.5]
$$

Therefore:

$$
X_{pos}
=
\begin{bmatrix}
0.3 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.4 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.8 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.5
\end{bmatrix}
$$

Shape:

$$
X_{pos} \in \mathbb{R}^{4 \times 4}
$$

At this stage:

- Each row contains semantic information from embeddings.
- Each row also contains positional information.
- The model now knows both token meaning and token order.
- No interaction between tokens has occurred yet.

The resulting matrix becomes the input to the self-attention mechanism.



# How QKV Matrices are Assigned Initially

The Query ($W_Q$), Key ($W_K$), and Value ($W_V$) matrices are the core learnable weights of the Attention mechanism. They project input embeddings into a new dimensional space. Unlike fixed positional encodings, these are **initialized as random numbers** and continuously updated via Gradient Descent during training.

#### 1. Initialization Distributions
At initialization, $W_Q$, $W_K$, and $W_V$ (typically of size `[hidden_dim, hidden_dim]`) are filled with random numbers drawn from specific statistical distributions to ensure stable training and prevent vanishing/exploding gradients:

* **Xavier/Glorot Uniform or Normal:** The most common default. It initializes weights so that the variance of outputs equals the variance of inputs.
* **Truncated Normal (GPT Style):** OpenAI often uses a normal distribution with a mean of 0 and a very small, specific standard deviation (e.g., `std = 0.02`).
* **Scaling by Dimension (Llama Style):** Meta often uses a normal distribution where the standard deviation is scaled by the model's dimension: `std = 1 / sqrt(hidden_dim)`.

#### 2. What About Biases?
* **Early Transformers:** The QKV linear layers included bias vectors ($b_Q, b_K, b_V$), which were initialized to **all zeros**.
* **Modern Trend:** Almost all modern LLMs (Llama, Mistral, Qwen) **remove biases entirely** from the attention layers to save compute and reduce parameter count. If present, they still start at exactly `0.0`.

#### 3. Special Initialization Tricks
The output projection matrix ($W_O$), which combines the attention heads back together, is sometimes initialized differently than $W_Q$, $W_K$, and $W_V$. 
* In some architectures, $W_O$ is initialized with **zeros** or a **much smaller variance**. 
* **Why?** This ensures the residual connections start by just passing the original input forward (acting like an identity function) before the model learns to attend to other words, stabilizing early training.

> **💡 Key Insight:** While positional encoding tells the model *where* a word is using fixed math, the QKV matrices learn *how much a word cares about other words* by starting as random noise and evolving through backpropagation.






## Query, Key and Value Generation

Three learnable weight matrices are initialized:

$$
W_Q=
\begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
W_K=
\begin{bmatrix}
1 & 1 \\
0 & 1 \\
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
W_V=
\begin{bmatrix}
1 & 0 \\
1 & 1 \\
0 & 1 \\
1 & 0
\end{bmatrix}
$$

Queries:

$$
Q=X_{pos}W_Q
$$

$$
Q=
\begin{bmatrix}
0.4 & 1.3 \\
1.3 & 0.5 \\
1.3 & 0.5 \\
0.8 & 1.4
\end{bmatrix}
$$

Keys:

$$
K=X_{pos}W_K
$$

$$
K=
\begin{bmatrix}
0.4 & 1.7 \\
1.3 & 1.8 \\
1.3 & 1.0 \\
0.8 & 2.3
\end{bmatrix}
$$

Values:

$$
V=X_{pos}W_V
$$

$$
V=
\begin{bmatrix}
1.6 & 0.9 \\
1.4 & 0.8 \\
1.0 & 1.0 \\
2.0 & 1.1
\end{bmatrix}
$$

## Attention Score Computation

Compute:

$$
QK^T
$$

For the token "sat":

$$
Q_{sat}
=
[1.3,0.5]
$$

Dot products:

$$
score(cat)=
(1.3)(0.4)+(0.5)(1.7)
=
1.37
$$

$$
score(sat)=
(1.3)(1.3)+(0.5)(1.8)
=
2.59
$$

$$
score(on)=
(1.3)(1.3)+(0.5)(1.0)
=
2.19
$$

$$
score(mat)=
(1.3)(0.8)+(0.5)(2.3)
=
2.19
$$

Score vector:

$$
[1.37,2.59,2.19,2.19]
$$

## Scaling

Since:

$$
d_k=2
$$

$$
\sqrt{d_k}=1.414
$$

Scaled scores:

$$
\frac{QK^T}{\sqrt{d_k}}
=
[0.97,1.83,1.55,1.55]
$$

## Softmax

Applying softmax:

$$
Softmax
([0.97,1.83,1.55,1.55])
$$

Produces:

$$
[0.15,0.35,0.25,0.25]
$$

Interpretation:

- 15% attention to cat
- 35% attention to sat
- 25% attention to on
- 25% attention to mat

## Weighted Value Aggregation

Attention output:

$$
Attention(Q,K,V)
=
Softmax(QK^T)V
$$

$$
=
0.15V_{cat}
+
0.35V_{sat}
+
0.25V_{on}
+
0.25V_{mat}
$$

Substituting values:

$$
=
0.15[1.6,0.9]
+
0.35[1.4,0.8]
+
0.25[1.0,1.0]
+
0.25[2.0,1.1]
$$

Result:

$$
[1.48,0.94]
$$

This becomes the new contextual representation for the token "sat".










## Feed Forward Network

The attention output is passed through:

$$
FFN(x)
=
GELU(xW_1+b_1)W_2+b_2
$$

For illustration:

$$
[1.48,0.94]
\rightarrow
[2.31,1.52]
$$

The Feed Forward Network refines and transforms contextual information.

## Output Projection

The final hidden state is projected into vocabulary space:

$$
logits = hW_{vocab}
$$

Example logits:

$$
mat = 8.2
$$

$$
chair = 2.1
$$

$$
floor = 1.4
$$

## Final Softmax

$$
P(token)
=
Softmax(logits)
$$

Result:

$$
mat = 0.84
$$

$$
chair = 0.10
$$

$$
floor = 0.06
$$

## Final Prediction

$$
argmax(P(token))
=
mat
$$

Therefore:

$$
\text{Prediction} = mat
$$

This completes the full Transformer forward pass from embeddings to next-token prediction for the sentence:

$$
\text{"cat sat on mat"}
$$

## Feed Forward Network (FFN)

The **Feed Forward Network (FFN)** is the second major component of a Transformer block, following the Multi-Head Attention layer.

While the attention mechanism enables tokens to exchange information with one another, the Feed Forward Network processes each token independently to learn more complex feature representations.

In simple terms:

- **Attention** determines **which information is important**.
- **Feed Forward Network** transforms and enriches that information.

### Why Do We Need a Feed Forward Network?

After the attention mechanism, each token has gathered contextual information from the entire sequence.

For example, after self-attention, the token:

> **"gave"**

already knows about:

- Alice
- Bob
- book

However, the attention mechanism only combines information from different tokens. It does not perform complex feature extraction.

The Feed Forward Network introduces additional non-linearity and learns richer representations of each token.

### Mathematical Formulation

The Feed Forward Network consists of two fully connected (linear) layers with a non-linear activation function in between.

The original Transformer uses:

$$
FFN(x)
=
W_2
\left(
ReLU(W_1x+b_1)
\right)
+b_2
$$

Modern LLMs typically replace ReLU with GELU or SwiGLU.

For example, GPT uses:

$$
FFN(x)
=
W_2
\left(
GELU(W_1x+b_1)
\right)
+b_2
$$

Llama uses:

$$
FFN(x)
=
W_2
\left(
SwiGLU(W_1x)
\right)
$$

### How Does It Work?

Suppose the attention output for the token **"gave"** is:

$$
x=
[1.48,\ 0.94]
$$

The first linear layer expands the hidden dimension.

Example:

$$
W_1:
2
\rightarrow
8
$$

The output becomes:

$$
[2.31,\ -0.84,\ 1.56,\ 0.42,\ 3.10,\ -1.25,\ 0.88,\ 2.74]
$$

Next, the activation function is applied.

For GELU:

$$
GELU(x)
$$

Negative values become smaller, while positive values are preserved.

The resulting vector is then projected back to the original hidden dimension using the second linear layer.

Example:

$$
8
\rightarrow
2
$$

Final output:

$$
[1.92,\ 1.14]
$$

### Why Expand the Hidden Dimension?

The first linear layer expands the feature space.

Example:

$$
512
\rightarrow
2048
$$

or

$$
4096
\rightarrow
11008
$$

This gives the model more capacity to learn complex feature interactions.

The second linear layer compresses the representation back to the original hidden dimension.

### Does the Feed Forward Network Mix Tokens?

No.

Unlike self-attention, the Feed Forward Network processes each token independently.

For example:

Before FFN:

$$
Alice
\rightarrow
[0.8,\ 1.1]
$$

$$
gave
\rightarrow
[1.5,\ 0.9]
$$

$$
Bob
\rightarrow
[0.7,\ 1.3]
$$

Each token passes through the same Feed Forward Network separately.

No interaction occurs between tokens inside the FFN.

### Position of FFN in a Transformer

The Transformer block follows this sequence:

Input

↓

Multi-Head Attention

↓

Residual Connection

↓

LayerNorm / RMSNorm

↓

Feed Forward Network

↓

Residual Connection

↓

LayerNorm / RMSNorm

↓

Output

### Important Points to Remember

- The Feed Forward Network comes after the attention mechanism.
- It processes each token independently.
- It consists of two linear layers.
- A non-linear activation function is applied between the two layers.
- The hidden dimension is first expanded and then projected back.
- It increases the expressive power of the model.
- It does not compute attention.
- The same FFN weights are shared across all tokens within the same Transformer layer.

### Interview Question

**Why do Transformers need a Feed Forward Network after the attention layer?**

**Answer:**

The attention mechanism enables tokens to exchange contextual information but performs only linear combinations of Value vectors. The Feed Forward Network introduces non-linearity and transforms each contextual representation independently, allowing the model to learn richer and more complex features before passing them to the next Transformer layer.

## Feed Forward Network (FFN)

After the self-attention mechanism completes, every token has gathered contextual information from all other tokens in the sequence.

For example, consider the sentence:

> **"Alice gave Bob a book."**

Initially, each token only represented its own semantic meaning.

After self-attention:

- **Alice** now knows she is the person performing the action.
- **gave** knows the relationship between Alice, Bob, and the book.
- **Bob** knows he is the receiver.
- **book** knows it is the object being transferred.

The output of the attention layer is therefore a set of **context-aware embeddings**.

A natural question arises:

> **If every token already contains contextual information, why do we need another layer?**

The answer is that **self-attention only mixes information between tokens**.

It does **not** perform complex feature extraction.

Self-attention computes a weighted average of Value vectors:

$$
Attention(Q,K,V)
=
Softmax
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

Notice that this equation only performs:

- Matrix multiplication
- Scaling
- Softmax
- Weighted averaging

The output is simply a linear combination of the Value vectors.

Although this allows tokens to exchange information, it does not create richer or more abstract feature representations.

For example,

suppose after attention the embedding for **"gave"** becomes

$$
[1.48,\ 0.94]
$$

This vector now contains contextual information, but it is still only a numerical representation.

The model has not yet learned higher-level concepts such as:

- subject-object relationships
- grammatical structure
- semantic composition
- abstract reasoning

To learn these richer representations, every token is passed through a **Feed Forward Network (FFN)**.

Unlike self-attention, the Feed Forward Network does **not** allow tokens to communicate.

Instead, it processes each token independently.

Its purpose is to transform the contextual embedding into a more expressive representation by applying multiple linear transformations and a non-linear activation function.

In other words,

- **Self-Attention decides which information each token should receive.**
- **Feed Forward Network decides how to transform that information into a richer representation.**

This combination of communication (Attention) and computation (FFN) forms the foundation of every Transformer block.

## Mathematical Formulation

The original Transformer uses a two-layer fully connected neural network.

The Feed Forward Network is defined as:

$$
FFN(x)
=
W_2
\left(
ReLU(W_1x+b_1)
\right)
+b_2
$$

where:

- \(W_1\) expands the hidden dimension.
- ReLU introduces non-linearity.
- \(W_2\) projects the representation back to the original hidden dimension.

Modern Large Language Models replace ReLU with more effective activation functions.

For example,

GPT models use:

$$
FFN(x)
=
W_2
\left(
GELU(W_1x+b_1)
\right)
+b_2
$$

Llama models use:

$$
FFN(x)
=
W_2
\left(
SwiGLU(W_1x)
\right)
$$

The overall architecture remains the same:

Input

↓

Linear Layer

↓

Activation Function

↓

Linear Layer

↓

Output

The Feed Forward Network is applied independently to every token in the sequence using the same set of learned weights.

If the input sequence contains \(n\) tokens, the FFN performs the same computation \(n\) times—once for each token.

Unlike self-attention, there is no interaction between different tokens inside the Feed Forward Network.

The communication between tokens has already occurred during the attention mechanism. The role of the Feed Forward Network is to enrich and transform each contextual representation before passing it to the next Transformer layer.

## Remaining Topics in a Transformer Block

### 1. Multi-Head Attention

Learn:

- Why one attention head is not enough
- Independent \(W_Q\), \(W_K\), and \(W_V\) for each head
- Parallel attention computation
- Concatenation of all heads
- Output projection using \(W_O\)

Formula:

$$
head_i
=
Attention(Q_i,K_i,V_i)
$$

$$
MultiHead
=
Concat(head_1,\ldots,head_h)W_O
$$

### 2. Output Projection

After concatenating all attention heads:

$$
O
=
Concat(head_1,\ldots,head_h)W_O
$$

Learn:

- Why \(W_O\) is required
- Shape of \(W_O\)
- Why the dimension is restored

### 3. Residual Connections

The attention output is added back to the original input.

Formula:

$$
Y=X+O
$$

Study:

- Skip connections
- Identity mapping
- Gradient flow

### 4. LayerNorm / RMSNorm

Normalize the residual output.

Formula:

$$
Y=
LayerNorm(X+O)
$$

or

$$
Y=
RMSNorm(X+O)
$$

Study:

- Why normalization comes after residual
- Pre-Norm vs Post-Norm
- Why Llama uses RMSNorm

### 5. Feed Forward Network

You have already completed this topic.

### 6. Second Residual Connection

After the FFN:

$$
Z
=
Y+FFN(Y)
$$

### 7. Final LayerNorm

Normalize the output before passing it to the next Transformer block.

### 8. Complete Transformer Block

Understand the complete architecture.

Pipeline:

Input

↓

RMSNorm

↓

Multi-Head Attention

↓

Residual Connection

↓

RMSNorm

↓

Feed Forward Network

↓

Residual Connection

↓

Output

### 9. Stacking Transformer Blocks

Study how multiple Transformer blocks are stacked.

Examples:

- BERT Base → 12 layers
- GPT-2 Large → 36 layers
- Llama 3 8B → 32 layers
- Llama 3 70B → 80 layers

### 10. Vocabulary Projection

Project the final hidden state into vocabulary space.

Formula:

$$
Logits
=
HW_{vocab}
$$

### 11. Softmax

Convert logits into probabilities.

### 12. Decoding

Study:

- Greedy Search
- Beam Search
- Temperature Sampling
- Top-k Sampling
- Top-p Sampling

### 13. Cross Entropy Loss

Formula:

$$
L
=
-\log(P(correct\ token))
$$

### 14. Backpropagation

Study how gradients are computed and propagated through the Transformer.

### 15. AdamW Optimizer

Study how all learnable parameters are updated during training.